In [1]:
!pip install transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.6 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving finetune_data.jsonl to finetune_data (1).jsonl


In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load dataset
dataset = load_dataset("json", data_files="finetune_data.jsonl")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # 🔥 ADD THIS LINE (IMPORTANT)
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(tokenize)

# Load model on GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)

# Training config
training_args = TrainingArguments(
    output_dir="./bhoomi_model",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"]
)

trainer.train()

# Save model
model.save_pretrained("bhoomi_lora")

Map:   0%|          | 0/3706 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
10,13.555096
20,9.560928
30,5.082247
40,3.191299
50,1.691092
60,1.220631
70,0.947563
80,1.107359
90,0.819231
100,0.934482
